# 第一阶段 步骤09：让函数更便利

> 来源：《深度学习入门2：自制框架》（斋藤康毅 著，郑明智 译，人民邮电出版社 2023）
> 目标：从零构建深度学习框架 **DeZero** 的第九步。

---

## 核心目标

框架已经能自动求导了，这一步做**易用性打磨**，共三处改进：

1. 让函数能像 Python 普通函数一样调用（`square(x)`）；
2. `backward` 自动初始化梯度（省去手动 `y.grad = 1`）；
3. 统一只处理 `ndarray`（用 `as_array` 兜底 + 类型检查）。

## 9.1 以 Python 函数形式调用

之前要先建函数对象再调用（`A = Square(); a = A(x)`），比较啰嗦。加一层包装函数后，就能直接 `square(x)`、`exp(x)`。

## 9.2 简化 backward

之前每次都要手动 `y.grad = np.array(1.0)` 再 `y.backward()`。现在 `backward` 开头自动判断：若 `grad` 为 `None`，就用 `np.ones_like(data)` 初始化为与数据同形状的全 1。

## 9.3 只处理 ndarray

`Variable` 被设计为只能存 `numpy.ndarray`。为兜底，引入 `as_array`：标量（`np.isscalar`）转成数组，数组原样返回。

再在 `Variable.__init__` 里加**类型检查**：不是 `ndarray` 就直接抛 `TypeError`，尽早暴露错误。

In [ ]:
import numpy as np

# —— 9.3 只处理 ndarray：as_array 把标量转成数组 ——
def as_array(x):
    if np.isscalar(x):
        return np.array(x)
    return x

class Variable:
    def __init__(self, data):
        if data is not None and not isinstance(data, np.ndarray):  # 9.3 类型检查
            raise TypeError(f'{type(data)} 不是支持的 ndarray 类型')
        self.data = data
        self.grad = None
        self.creator = None

    def set_creator(self, func):
        self.creator = func

    def backward(self):                        # 9.2 自动初始化梯度
        if self.grad is None:
            self.grad = np.ones_like(self.data)
        funcs = [self.creator]
        while funcs:
            f = funcs.pop()
            x, y = f.input, f.output
            x.grad = f.backward(y.grad)
            if x.creator is not None:
                funcs.append(x.creator)

class Function:
    def __call__(self, input):
        x = input.data
        y = self.forward(x)
        output = Variable(as_array(y))         # 9.3 输出统一为 ndarray
        output.set_creator(self)
        self.input = input
        self.output = output
        return output

    def forward(self, x):
        raise NotImplementedError()

    def backward(self, gy):
        raise NotImplementedError()

class Square(Function):
    def forward(self, x):
        return x ** 2
    def backward(self, gy):
        x = self.input.data
        return 2 * x * gy

class Exp(Function):
    def forward(self, x):
        return np.exp(x)
    def backward(self, gy):
        x = self.input.data
        return np.exp(x) * gy

# —— 9.1 以 Python 函数形式调用 ——
def square(x):
    return Square()(x)

def exp(x):
    return Exp()(x)

In [ ]:
# 连续调用，代码更简洁
x = Variable(np.array(0.5))
y = square(exp(square(x)))   # y = (e^(x^2))^2
y.backward()                 # 无需手动设置 y.grad = 1

print(x.grad)   # 3.297442541400256

## 这一步的"为什么"

- **`as_array` / 类型检查**：把错误拦截在数据入口，避免标量混进数组运算导致难以排查的 bug；
- **`square` / `exp` 包装**：让框架 API 更接近 PyTorch 的 `F.square` 风格，体验更好；
- **自动初始化梯度**：反向传播最常见的场景就是"求整个输出的梯度"，默认从全 1 开始最符合直觉。

---

> 预告：步骤10 用 `unittest` 给框架补上**测试**，第一阶段「自动微分」就此收官。